# 03 — Distributed compute (CodeFlare RayJob)

Run `distributed_stats.py` on a Ray cluster. Each worker summarizes a partition of the Iris dataset using `@ray.remote` tasks.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "extras/python"))
from workshop_common import (
    LOCAL_QUEUE,
    NAMESPACE,
    RayJob,
    cpu_cluster_config,
    login,
    runtime_env_for_script,
    submit_rayjob,
    wait_for_rayjob,
)

In [ ]:
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN".strip()
auth = login(OPENSHIFT_TOKEN, OPENSHIFT_SERVER)

In [ ]:
job = RayJob(
    job_name="ray-workshop-distributed-stats",
    entrypoint="python extras/scripts/distributed_stats.py",
    cluster_config=cpu_cluster_config(num_workers=2),
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    runtime_env=runtime_env_for_script(
        env_vars={
            "INPUT_PATH": "extras/data/iris.csv",
            "NUM_PARTITIONS": "4",
        },
    ),
    ttl_seconds_after_finished=600,
)
job_name = submit_rayjob(job)

In [ ]:
wait_for_rayjob(job_name)

## Check results

```sh
oc logs -n ray-workshop -l ray.io/node-type=head -c ray-head --tail=100 | tail -30
```

Expect JSON partition summaries and `Aggregated row count: 30`.